In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert ".venv" in sys.executable, "Wrong kernel! Switch to 'Python (f1-tyre-strategy)' (top right)."
PROJECT_ROOT = Path.cwd().parent
assert PROJECT_ROOT.name == "f1-tyre-strategy-simulator", "Notebook isn't in the notebooks/ folder!"

RACES_DIR = PROJECT_ROOT / "data" / "raw" / "races"
QUALI_DIR = PROJECT_ROOT / "data" / "raw" / "quali"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DRY = ["SOFT", "MEDIUM", "HARD"]
WET = ["INTERMEDIATE", "WET"]
print("✅ Kernel and paths OK")

✅ Kernel and paths OK


In [2]:
laps = pd.concat([pd.read_parquet(f) for f in sorted(RACES_DIR.glob("*.parquet"))], ignore_index=True)
quali = pd.concat([pd.read_parquet(f) for f in sorted(QUALI_DIR.glob("*.parquet"))], ignore_index=True)

n_races = laps.groupby(["Year", "Round"]).ngroups
n_quali = quali.groupby(["Year", "Round"]).ngroups
print(f"Race laps: {laps.shape[0]:,} rows × {laps.shape[1]} columns | races: {n_races}")
print(f"Qualifying: {quali.shape[0]:,} driver rows | sessions: {n_quali}")

assert n_races == 92 and n_quali == 92, "Expected 92 races and 92 qualifying sessions"
print("✅ Both sources complete")

Race laps: 101,291 rows × 30 columns | races: 92
Qualifying: 1,831 driver rows | sessions: 92
✅ Both sources complete


In [3]:
# FastF1 renamed Miami's location in 2025; map it back so it's one circuit
CIRCUIT_FIXES = {"Miami Gardens": "Miami"}
laps["Circuit"] = laps["Location"].replace(CIRCUIT_FIXES)

races = laps.drop_duplicates(["Year", "Round"])[["Year", "Round", "EventName", "Circuit"]]
counts = races["Circuit"].value_counts()
print("Races per circuit:")
print(counts.to_string())

assert races["Circuit"].nunique() == 25, f"Expected 25 circuits, found {races['Circuit'].nunique()}"
assert counts["Miami"] == 4, "Miami should have 4 races (2022–2025)"
print("\n✅ 25 circuits; Miami now has 4 races")

Races per circuit:
Circuit
Sakhir               4
Jeddah               4
Yas Island           4
São Paulo            4
Mexico City          4
Austin               4
Suzuka               4
Marina Bay           4
Monza                4
Zandvoort            4
Spa-Francorchamps    4
Budapest             4
Spielberg            4
Silverstone          4
Montréal             4
Baku                 4
Monaco               4
Barcelona            4
Miami                4
Melbourne            4
Imola                3
Lusail               3
Las Vegas            3
Shanghai             2
Le Castellet         1

✅ 25 circuits; Miami now has 4 races


In [4]:
quali_cols = ["Year", "Round", "Driver", "BestQualiLapSeconds", "BestQualiCompound",
              "QualiWetTyreShare", "QualiRainfall"]

# One row per driver per race, with the results-table time from the race files
drivers = laps.drop_duplicates(["Year", "Round", "Driver"])[
    ["Year", "Round", "EventName", "Circuit", "Driver", "Team", "BestQualiSeconds"]
]
check = drivers.merge(quali[quali_cols], on=["Year", "Round", "Driver"], how="left")

both = check.dropna(subset=["BestQualiSeconds", "BestQualiLapSeconds"]).copy()
both["Diff"] = (both["BestQualiLapSeconds"] - both["BestQualiSeconds"]).round(3)

print(f"Race drivers: {len(check):,} | with both qualifying sources: {len(both):,}")
print(f"Sources identical (within 0.01 s): {(both['Diff'].abs() <= 0.01).mean():.1%}")
print(f"Differ by more than 0.1 s: {(both['Diff'].abs() > 0.1).sum()}")
print(both.loc[both["Diff"].abs() > 0.1,
               ["Year", "EventName", "Driver", "BestQualiSeconds", "BestQualiLapSeconds", "Diff"]]
      .sort_values("Diff").head(15).to_string(index=False))

print("\nOnly the results-table time:", (check["BestQualiSeconds"].notna() & check["BestQualiLapSeconds"].isna()).sum())
print("Only the lap-data time:     ", (check["BestQualiSeconds"].isna() & check["BestQualiLapSeconds"].notna()).sum())
print("Neither:                    ", (check["BestQualiSeconds"].isna() & check["BestQualiLapSeconds"].isna()).sum())

Race drivers: 1,827 | with both qualifying sources: 1,789
Sources identical (within 0.01 s): 97.1%
Differ by more than 0.1 s: 41
 Year                 EventName Driver  BestQualiSeconds  BestQualiLapSeconds    Diff
 2023  Saudi Arabian Grand Prix    SAR           128.510               89.721 -38.789
 2024      São Paulo Grand Prix    ALO            88.998               85.035  -3.963
 2023        Belgian Grand Prix    RIC           122.159              119.483  -2.676
 2025  United States Grand Prix    ALB            94.690               93.670  -1.020
 2025 Emilia Romagna Grand Prix    BEA            76.918               76.077  -0.841
 2022       Austrian Grand Prix    PER            66.143               65.404  -0.739
 2022    Mexico City Grand Prix    MSC            80.419               79.695  -0.724
 2025  United States Grand Prix    STR            94.540               93.878  -0.662
 2024     Australian Grand Prix    RIC            78.085               77.466  -0.619
 2023      

In [5]:
ref = check.copy()

# 1st choice: official results-table time
ref["QualiRef"] = ref["BestQualiSeconds"]
ref["QualiRefSource"] = np.where(ref["QualiRef"].notna(), "results", None)

# 2nd choice: best valid lap from the qualifying lap data (fixes 2025 Miami)
use_lap = ref["QualiRef"].isna() & ref["BestQualiLapSeconds"].notna()
ref.loc[use_lap, "QualiRef"] = ref.loc[use_lap, "BestQualiLapSeconds"]
ref.loc[use_lap, "QualiRefSource"] = "lap_data"

# 3rd choice: team-mate's reference (same car, same session)
teammate_ref = ref.groupby(["Year", "Round", "Team"])["QualiRef"].transform("min")
use_tm = ref["QualiRef"].isna() & teammate_ref.notna()
ref.loc[use_tm, "QualiRef"] = teammate_ref[use_tm]
ref.loc[use_tm, "QualiRefSource"] = "teammate"

print("Where each driver's qualifying reference comes from:")
print(ref["QualiRefSource"].value_counts(dropna=False).to_string())
print("\nStill missing:")
print(ref.loc[ref["QualiRef"].isna(), ["Year", "EventName", "Driver", "Team"]].to_string(index=False))

# Attach to every lap (new columns), checking no rows are gained or lost
n_before = len(laps)
laps = laps.merge(ref[["Year", "Round", "Driver", "QualiRef", "QualiRefSource"]],
                  on=["Year", "Round", "Driver"], how="left")
assert len(laps) == n_before, "Merge changed the number of laps!"
print(f"\n✅ QualiRef attached to {laps['QualiRef'].notna().mean():.1%} of laps")

Where each driver's qualifying reference comes from:
QualiRefSource
results     1789
lap_data      31
teammate       7

Still missing:
Empty DataFrame
Columns: [Year, EventName, Driver, Team]
Index: []

✅ QualiRef attached to 100.0% of laps


In [6]:
# Wet race: any race laps on intermediate or wet tyres
race_flags = (
    laps.groupby(["Year", "Round", "EventName", "Circuit"])["Compound"]
        .apply(lambda c: c.isin(WET).mean())
        .rename("RaceWetTyreShare").reset_index()
)

# Wet qualifying: any qualifying laps on intermediate or wet tyres (rain alone doesn't count)
session = quali.drop_duplicates(["Year", "Round"])[["Year", "Round", "QualiWetTyreShare", "QualiRainfall"]]
race_flags = race_flags.merge(session, on=["Year", "Round"], how="left")

race_flags["WetRace"] = race_flags["RaceWetTyreShare"] > 0
race_flags["WetQuali"] = race_flags["QualiWetTyreShare"] > 0
race_flags["ExcludeRace"] = race_flags["WetRace"] | race_flags["WetQuali"]

print(f"Wet race:   {race_flags['WetRace'].sum()}")
print(f"Wet quali:  {race_flags['WetQuali'].sum()}")
print(f"Both:       {(race_flags['WetRace'] & race_flags['WetQuali']).sum()}")
print(f"Excluded:   {race_flags['ExcludeRace'].sum()} of {len(race_flags)}")

print("\nRaces per season (total / excluded / kept):")
season = race_flags.groupby("Year").agg(Total=("ExcludeRace", "size"), Excluded=("ExcludeRace", "sum"))
season["Kept"] = season["Total"] - season["Excluded"]
print(season.to_string())

print("\nExcluded races and why:")
excl = race_flags[race_flags["ExcludeRace"]].copy()
excl["Reason"] = np.select(
    [excl["WetRace"] & excl["WetQuali"], excl["WetRace"]],
    ["wet race + wet quali", "wet race"], default="wet quali",
)
print(excl[["Year", "EventName", "Reason"]].to_string(index=False))

# --- Cross-check only (not used as a rule): median race-vs-quali gap on rough clean laps ---
rough = laps[
    laps["Compound"].isin(DRY) & laps["PitInTime"].isna() & laps["PitOutTime"].isna()
    & (laps["LapNumber"] > 1) & laps["IsAccurate"].eq(True)
    & laps["TrackStatus"].astype(str).eq("1") & laps["QualiRef"].notna()
]
median_gap = (
    (rough["LapTimeSeconds"] - rough["QualiRef"])
    .groupby([rough["Year"], rough["Round"]]).median()
    .rename("MedianGap").reset_index()
)
race_flags = race_flags.merge(median_gap, on=["Year", "Round"], how="left")

print("\nCross-check — median gap in races flagged for wet qualifying:")
print(race_flags.loc[race_flags["WetQuali"], ["Year", "EventName", "MedianGap"]]
      .sort_values("MedianGap").round(2).to_string(index=False))
print("\nCross-check — lowest median gaps among races we KEEP:")
print(race_flags.loc[~race_flags["ExcludeRace"], ["Year", "EventName", "QualiRainfall", "MedianGap"]]
      .nsmallest(8, "MedianGap").round(2).to_string(index=False))

Wet race:   12
Wet quali:  12
Both:       5
Excluded:   19 of 92

Races per season (total / excluded / kept):
      Total  Excluded  Kept
Year                       
2022     22         7    15
2023     22         4    18
2024     24         4    20
2025     24         4    20

Excluded races and why:
 Year                 EventName               Reason
 2022 Emilia Romagna Grand Prix wet race + wet quali
 2022         Monaco Grand Prix             wet race
 2022       Canadian Grand Prix            wet quali
 2022        British Grand Prix            wet quali
 2022      Singapore Grand Prix wet race + wet quali
 2022       Japanese Grand Prix             wet race
 2022      São Paulo Grand Prix            wet quali
 2023         Monaco Grand Prix             wet race
 2023       Canadian Grand Prix            wet quali
 2023        Belgian Grand Prix            wet quali
 2023          Dutch Grand Prix wet race + wet quali
 2024       Canadian Grand Prix             wet race
 2024   